In [1]:
## The following code ensures that all functions and init files are reloaded before executions.
%load_ext autoreload
%autoreload 2

In [2]:
import dask
dask.config.set({'dataframe.query-planning': False})
import spatialdata

c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\spatialdata\_core\query\relational_query.py:530: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap 

In [3]:
from insitupy.datasets import xenium_human_breast_cancer, visium_human_breast_cancer

c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
xenium = xenium_human_breast_cancer()
xenium.load_all()

This dataset exists already. Download is skipped. To force download set `overwrite=True`.
Image exists. Checking md5sum...
The md5sum matches. Download is skipped. To force download set `overwrite=True`.
Image exists. Checking md5sum...
The md5sum matches. Download is skipped. To force download set `overwrite=True`.
Corresponding image data can be found in C:\Users\ge37voy\.cache\InSituPy\demo_datasets\hbreastcancer\unregistered_images
For this dataset following images are available:
slide_id__hbreastcancer__HE__histo.ome.tiff
slide_id__hbreastcancer__CD20_HER2_DAPI__IF.ome.tiff
Loading cells...
Loading images...
Loading transcripts...


In [ ]:
from insitupy.io import read_xenium

In [ ]:
p = r"C:\Users\ge37voy\.cache\InSituPy\demo_datasets\hbreastcancer\output-XETG00000__slide_id__hbreastcancer"
x = read_xenium(p, backend="spatialdata")

INFO: Reading Xenium data with spatialdata-io...
c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Extracted pixel size 0.2125 from 'cell_circles'
Adding images...
Adding cell data...


Adding transcripts...


In [8]:
x

InSituData
Method:		Xenium
Slide ID:	slide_test
Sample ID:	sample_test
Path:		None

    ➤ images
       nuclei:	(1, 25778, 35416)
       mip:	(1, 25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           matrix
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region'
               var: 'gene_ids', 'feature_types', 'genome'
               uns: 'spatialdata_attrs'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
                   nuclei
    ➤ transcripts
       DataFrame with shape Delayed('int-2e98e1f0-d156-4ea0-a9a2-4d8e6d2a7fd7') x 8

In [ ]:
visium = visium_human_breast_cancer()

In [4]:
from spatialdata_io import visium

vp = r"C:\Users\ge37voy\.cache\InSituPy\demo_datasets\visium_hbreastcancer\CytAssist_FFPE_Human_Breast_Cancer"
vis = visium(vp, dataset_id="bc")

c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\anndata\_core\anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\anndata\_core\anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\ge37voy\AppData\Local\miniconda3\envs\sd313\Lib\site-packages\spatialdata\models\models.py:1183: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


In [5]:
vis

SpatialData object
├── Images
│     ├── 'bc_hires_image': DataArray[cyx] (3, 2000, 1809)
│     └── 'bc_lowres_image': DataArray[cyx] (3, 600, 543)
├── Shapes
│     └── 'bc': GeoDataFrame shape: (4992, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (4992, 18085)
with coordinate systems:
    ▸ 'bc', with elements:
        bc_hires_image (Images), bc_lowres_image (Images), bc (Shapes)
    ▸ 'bc_downscaled_hires', with elements:
        bc_hires_image (Images), bc (Shapes)
    ▸ 'bc_downscaled_lowres', with elements:
        bc_lowres_image (Images), bc (Shapes)

In [6]:
from insitupy.spatialdata.convert import convert_from_spatialdata

In [7]:
dataset_id = "bc"
sample_id = "test_id"
# Convert to InSituData format
data = convert_from_spatialdata(
    sdata=vis,
    image_keys={
        "hires": f"{dataset_id}_hires_image",
        "lowres": f"{dataset_id}_lowres_image"
    },
    features_key=dataset_id,
    cells_key=None,  # No cells available in Visium data
    table_key="table",
    cell_boundaries_key=None,  # Visium uses spots, not boundaries
    nucleus_boundaries_key=None,
    transcripts_key=None,  # Visium doesn't have single-molecule transcripts
    slide_id=dataset_id if dataset_id else "visium_slide",
    sample_id=sample_id if sample_id else "visium_sample",
    method_name="Visium",
)

INFO: Using 'bc' coordinate system for pixel size extraction.


Extracted pixel size 1.0 from 'bc'
Adding images...


In [8]:
data

InSituData
Method:		Visium
Slide ID:	bc
Sample ID:	test_id
Path:		None

    ➤ images
       'hires':    (3, 2000, 1809)
       'lowres':   (3, 600, 543)
    ➤ features
       FeatureData (Type: 'feature')
           .data: 4992 obs × 18085 vars
           .shapes: 4992 geometries

In [9]:
from insitupy import CACHE

In [10]:
feature_path = CACHE / "out" / "features_test"
data.saveas(feature_path, overwrite=True)

Saving data to C:\Users\ge37voy\.cache\InSituPy\out\features_test


... storing 'feature_types' as categorical
... storing 'genome' as categorical


Saved.


In [13]:
from insitupy import InSituData

In [14]:
vr = InSituData.read(feature_path)

In [16]:
vr.load_all()

In [17]:
vr

InSituData
Method:		Visium
Slide ID:	bc
Sample ID:	test_id
Path:		C:\Users\ge37voy\.cache\InSituPy\out\features_test

    ➤ images
       'hires':    (3, 2000, 1809)
       'lowres':   (3, 600, 543)
    ➤ features
       FeatureData (Type: 'feature')
           .data: 4992 obs × 18085 vars
           .shapes: 4992 geometries

In [12]:
data.features.pixel_size

1.0

In [20]:
for elem_type, key, elem in vis.gen_elements():
    print(key)

bc_hires_image
bc_lowres_image
bc
table


In [ ]:
xdconv = convert_from_spatialdata(
    sdata=sdata,
    image_keys={
        "nuclei": "morphology_mip",
        "mip": "morphology_focus"
    },
    cells_key="cell_circles",
    table_key="table",
    cell_boundaries_key="cell_labels",
    nucleus_boundaries_key="nucleus_labels",
    transcripts_key="transcripts",
    slide_id="slide_test",
    sample_id="sample_test",
    method_name="Xenium"
)